# Jupyter Notebook
# Lect 16: Subset Selection

In [83]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [84]:
# First, we're going to do all the data loading we've had for a while for this data set
auto = pd.read_csv('../../DataSets/Auto.csv')
auto = auto.replace('?', np.nan)
auto = auto.dropna()
auto.horsepower = auto.horsepower.astype('int')

#this shuffles my data set in advance so that i don't need to worry about it later 
auto = auto.sample(frac=1).reset_index(drop=True)


auto.head()


,mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
0,27.0,4,140.0,86,2790,15.6,82,1,ford mustang gl
1,19.0,6,232.0,100,2634,13.0,71,1,amc gremlin
2,34.1,4,91.0,68,1985,16.0,81,3,mazda glc 4
3,30.0,4,146.0,67,3250,21.8,80,2,mercedes-benz 240d
4,24.0,4,90.0,75,2108,15.5,74,2,fiat 128


Let's try to run subset selection on the `auto` data set! We're going to use `cylinders`, `horsepower`, `weight`, and `acceleration` to predict `mpg`. 

In [85]:
inputvars = ['cylinders','horsepower','weight', 'acceleration']

The first tool we are going to use is the `itertools` package, which gives us a way to get subsets of whatever size we want using the `combinations` command.  

In [86]:
from itertools import combinations

The weird thing is it's an iterator, so if I just try to print out what I want, it's not helpful to me. 

In [87]:
combinations(inputvars,2)

But if I use it in a for loop it does what I want!

In [88]:
for x in combinations(inputvars,2):
    print(x)

('cylinders', 'horsepower')
('cylinders', 'weight')
('cylinders', 'acceleration')
('horsepower', 'weight')
('horsepower', 'acceleration')
('weight', 'acceleration')


Here's some code stolen from the last few days to run linear regression on a subset of the input variables and get the *training error*. Note that this has no train/test split.  

In [89]:
def myscore(df,listofvars, outputvar = 'mpg'):
    '''
    This code returns the training mean squared error of a linear regression model. No train test split or anything like that. 
    '''
    if listofvars == 'null model':
        # Predict the mean of the output variable for any input 
        yhat = np.full(auto.shape[0],np.average(auto.mpg))
        return mean_squared_error(auto.mpg, yhat)
    
    else:
        X = df[list(listofvars)]
        y = df[outputvar]
        
        #build linear regression model
        model = LinearRegression()
        
        # fit model
        model.fit(X,y)

        #predict y
        ypred = model.predict(X)

        #view mean absolute error
        return mean_squared_error(y,ypred)
    

myvars = ('cylinders', 'acceleration')
print(f"MSE for the (cylinders, acceleration) model: {myscore(auto,myvars)}")

print(f"MSE for the null model: {myscore(auto,'null model')}")

MSE for the (cylinders, acceleration) model: 23.94244665060135
MSE for the null model: 60.7627384423157



&#9989; **<font color=red>Do this:</font>** Modify the code below by  using the `myscore` function to get the training error for each possible model of size $k=2$.


In [90]:
def getBestModel_df(df, k=2, outputvar = 'mpg'):
    '''
    This function will return the best model for a given number of input variables
    '''
    myvars = []
    myscores = []
    
    if k == 0:
        return pd.DataFrame({'Vars':'null model', 'Train Score':[myscore(auto,'null model')]})
    else:

        for Xs in combinations(inputvars,k):
            myvars.append(Xs) 
            myscores.append(None) #<--- Fix this None
                
        myResults = pd.DataFrame({'Vars':myvars, 'Train Score':myscores})
        return myResults

print("k = 2")
print(getBestModel_df(auto, k=2))
print("\n")
print("k = 0, so this is the null model")
print(getBestModel_df(auto, k=0))

k = 2
                         Vars Train Score
0     (cylinders, horsepower)        None
1         (cylinders, weight)        None
2   (cylinders, acceleration)        None
3        (horsepower, weight)        None
4  (horsepower, acceleration)        None
5      (weight, acceleration)        None


k = 0, so this is the null model
         Vars  Train Score
0  null model    60.762738


In [91]:
##ANSWER##
def getBestModel_df(df, k=2, outputvar = 'mpg'):
    '''
    This function will return the best model for a given number of input variables
    '''
    myvars = []
    myscores = []
    
    if k == 0:

        return pd.DataFrame({'Vars':'null model', 'Train Score':[myscore(auto,'null model')]})
    else:

        for Xs in combinations(inputvars,k):
            myvars.append(Xs) 

            myscores.append(myscore(auto,Xs)) #<--- Fix this None
                
        myResults = pd.DataFrame({'Vars':myvars, 'Train Score':myscores})
        return myResults

print("k = 2")
print(getBestModel_df(auto, k=2))
print("\n")
print("k = 0, so this is the null model")
print(getBestModel_df(auto, k=0))

k = 2
                         Vars  Train Score
0     (cylinders, horsepower)    20.848190
1         (cylinders, weight)    18.382946
2   (cylinders, acceleration)    23.942447
3        (horsepower, weight)    17.841442
4  (horsepower, acceleration)    22.461644
5      (weight, acceleration)    18.247176


k = 0, so this is the null model
         Vars  Train Score
0  null model    60.762738



&#9989; **<font color=red>Q:</font>** What is $M_2$? That is, what is the set of variables that give the lowest training error over all sets of size 2? Write a command to pull this automatically from the table above. 

*Hint: the command `DataFrame.idxmin()` will give you the row index of the minimum entry in a pandas series.*


In [92]:
# Your code here 

def getBestModel(df, k=2, outputvar = 'mpg'):
    '''
    This function will return the best model for a given number of input variables
    '''
    myResults = getBestModel_df(df, k, outputvar)
    
    # Add code here to find the index of the minimum training score
    M_k = None
    return M_k

print(getBestModel(auto, k=2))

# Check to make sure this is right :-P
getBestModel(auto, k=2) == ('horsepower', 'weight')

None


False

In [93]:
##ANSWER##
def getBestModel(df, k=2, outputvar = 'mpg'):
    '''
    This function will return the best model for a given number of input variables
    '''
    myResults = getBestModel_df(df, k, outputvar)
    
    # Find the index of the minimum training score
    indexmin = myResults['Train Score'].idxmin()
    M_k = myResults.Vars[indexmin]
    return M_k

print(getBestModel(auto, k=2))

# Check to make sure this is right :-P
getBestModel(auto, k=2) == ('horsepower', 'weight')

('horsepower', 'weight')


True

Now, once we've found all the $M_k$, we want to return the best model using the *testing* error. So, here's a function that will take in a list of variables and give you the 5-fold CV score. 

In [94]:
def myscore_cv(df,listofvars, outputvar = 'mpg'):


    if listofvars == 'null model':
        # Predict the mean of the output variable for any input
        test_scores = [] 
        for (train,test) in KFold(5).split(df):
            y_test = df[outputvar].iloc[train]
            yhat = np.full(y_test.shape[0],np.average(y_test))
            test_scores.append(mean_squared_error(y_test, yhat))
        return np.average(test_scores)
    else:
        X = df[list(listofvars)]
        y = df[outputvar]
        #build linear regression model
        model = LinearRegression()
        
        #use 5-fold CV to evaluate model
        scores = cross_val_score(model, X,y, 
                                scoring='neg_mean_squared_error',
                                cv=5)

        #view mean absolute error
        return np.average(np.absolute(scores))
    

myvars = ('cylinders', 'acceleration')
myscore_cv(auto,myvars)

24.958636976147925


&#9989; **<font color=red>Do this:</font>** Put it all together. Finish the for loop that runs through $k=0$ to $k=5$, and gets the set of variables giving the best size $k$ model, and then computs the test error for each model. Which model would be returned by the best subset selection method? 


In [95]:

for k in range(5):
    M_k = None #<-- Use the getBestModel function here to get the list of variables
    print(f"Best {k} variable model: {M_k}")
    M_k_CV = 0 #<-- Use the myscore_cv function here to get the test error
    print(f"CV Score: {M_k_CV:0.3f}")


Best 0 variable model: None
CV Score: 0.000
Best 1 variable model: None
CV Score: 0.000
Best 2 variable model: None
CV Score: 0.000
Best 3 variable model: None
CV Score: 0.000
Best 4 variable model: None
CV Score: 0.000


In [99]:
##ANSWER##
for k in range(5):
    M_k = getBestModel(auto, k)
    print(f"Best {k} variable model: {M_k}")
    M_k_CV = myscore_cv(auto,M_k)
    print(f"CV Score: {M_k_CV:0.3f}")

# The best model is the 2 variable model using horsepower and weight

Best 0 variable model: null model
CV Score: 60.743
Best 1 variable model: ('weight',)
CV Score: 19.104
Best 2 variable model: ('horsepower', 'weight')
CV Score: 18.286
Best 3 variable model: ('cylinders', 'horsepower', 'weight')
CV Score: 18.407
Best 4 variable model: ('cylinders', 'horsepower', 'weight', 'acceleration')
CV Score: 18.800




-----
### Congratulations, we're done!
Initially created by Dr. Liz Munch, modified by Dr. Lianzhang Bao, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.

In [97]:
##ANSWER##
#This cell runs the converter which removes ANSWER fields, renames the notebook and cleans out output fields. 

from jupyterinstruct import InstructorNotebook
import os
this_notebook = os.path.basename(globals()['__vsc_ipynb_file__'])

studentnotebook = InstructorNotebook.makestudent(this_notebook)

InstructorNotebook.validate(studentnotebook)


Myfilename CMSE381-Lec17_Subset-INSTRUCTOR.ipynb


CMSE381-Lec17_Subset.ipynb


Validating Notebook ./CMSE381-Lec17_Subset.ipynb


0